# StormEngine — Stage 1: Encoder (SetConv)

This notebook implements the **SetConv Encoder** — Stage 1 of the StormEngine pipeline.

## Purpose
Convert sparse, irregular point observations into a consistent gridded representation `(C, H, W)` ready for the Vision Transformer Processor.

## Pre-training strategy
Since DPC FVG station data has no historical archive, pre-training is performed entirely on ERA5:
- **Input (sparse):** ERA5 standard variables (`msl`, `u10`, `v10`, `i10fg`) extracted at fixed station coordinates
- **Target (grid):** ERA5-Land variables (`t2m`, `tp`) as reconstruction ground truth

## Input files
| File | Content | Role |
|---|---|---|
| `data_0.nc` | ERA5-Land: `t2m` [K], `tp` [m] @ 9 km | **Target grid** |
| `data_stream-oper_stepType-instant.nc` | ERA5 standard: `msl`, `u10`, `v10`, `i10fg` @ 31 km | **Sparse input source** |
| `value_lat_lon_temp.csv` | ERA5 grid values `(VALUE, lon, lat)` | **Fixed station coordinate pool** |

---
## 0. Imports and Configuration

In [3]:
import numpy as np
import pandas as pd
import xarray as xr
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Paths
PATH_ERA5_LAND = 'data_0.nc'
PATH_ERA5_STD  = 'data_stream-oper_stepType-instant.nc'
PATH_CSV       = 'value_lat_lon_temp.csv'

# ── Domain bounding box (North-East Adriatic / FVG)
LAT_MIN, LAT_MAX = 39.0, 46.5
LON_MIN, LON_MAX = 12.0, 20.0

# ── Variables
VARS_INPUT   = ['msl', 'u10', 'v10', 'i10fg']   # ERA5 standard  -> sparse input
VARS_TARGET  = ['t2m', 'tp']                     # ERA5-Land      -> grid target
N_INPUT_VARS  = len(VARS_INPUT)
N_TARGET_VARS = len(VARS_TARGET)

# ── SetConv fixed station grid
N_STATION_POINTS = 20    # mirrors DPC FVG coastal station density

# ── Training hyperparameters
BATCH_SIZE    = 4
LEARNING_RATE = 1e-4
N_EPOCHS      = 30
HIDDEN_DIM    = 64
KERNEL_SIZE   = 5
SIGMA         = 0.15     # Gaussian pooling bandwidth

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device        : {DEVICE}')
print(f'Input vars    : {VARS_INPUT}')
print(f'Target vars   : {VARS_TARGET}')

Device        : cpu
Input vars    : ['msl', 'u10', 'v10', 'i10fg']
Target vars   : ['t2m', 'tp']


---
## 1. Load ERA5 Data

In [4]:
# ERA5-Land (TARGET)
ds_land = xr.open_dataset(PATH_ERA5_LAND)
print('=== ERA5-Land ===')
print(ds_land)
for v in ds_land.data_vars:
    da = ds_land[v]
    print(f'  {v}: shape={da.shape}  units={da.attrs.get("units","?")}  '
          f'range=[{float(da.min()):.3f}, {float(da.max()):.3f}]')

FileNotFoundError: [Errno 2] No such file or directory: 'd:\\StormEngine\\modelML\\data_0.nc'

In [ ]:
# ERA5 standard (INPUT)
ds_std = xr.open_dataset(PATH_ERA5_STD)
print('=== ERA5 standard ===')
print(ds_std)
for v in ds_std.data_vars:
    da = ds_std[v]
    print(f'  {v}: shape={da.shape}  units={da.attrs.get("units","?")}  '
          f'range=[{float(da.min()):.3f}, {float(da.max()):.3f}]')

In [ ]:
# CSV — ERA5 grid with lat/lon (used to define fixed station coords)
df_csv = pd.read_csv(PATH_CSV)
print('=== CSV ===')
print(f'Shape: {df_csv.shape}')
print(f'Columns: {list(df_csv.columns)}')
print(f'Lat: {df_csv["lat"].min()} — {df_csv["lat"].max()}')
print(f'Lon: {df_csv["lon"].min()} — {df_csv["lon"].max()}')
print(f'VALUE (t2m K): {df_csv["VALUE"].min():.2f} — {df_csv["VALUE"].max():.2f}')
df_csv.head()

---
## 2. Unit Conversions

| Variable | Raw unit | Target unit | Formula |
|---|---|---|---|
| `t2m` | K | °C | `− 273.15` |
| `tp` | m | mm | `× 1000` |
| `msl` | Pa | hPa | `÷ 100` |
| `u10`, `v10`, `i10fg` | m/s | m/s | — |

In [ ]:
def convert_land(ds):
    ds = ds.copy()
    if 't2m' in ds:
        ds['t2m'] = ds['t2m'] - 273.15
        ds['t2m'].attrs['units'] = 'C'
    if 'tp' in ds:
        ds['tp'] = ds['tp'] * 1000
        ds['tp'].attrs['units'] = 'mm'
    return ds

def convert_std(ds):
    ds = ds.copy()
    if 'msl' in ds:
        ds['msl'] = ds['msl'] / 100
        ds['msl'].attrs['units'] = 'hPa'
    return ds

ds_land = convert_land(ds_land)
ds_std  = convert_std(ds_std)

# CSV column
df_csv['t2m_celsius'] = df_csv['VALUE'] - 273.15

print('Unit conversions applied.')
for v in ds_land.data_vars:
    da = ds_land[v]
    print(f'  land/{v}: [{float(da.min()):.3f}, {float(da.max()):.3f}] {da.attrs.get("units","?")}')
for v in ds_std.data_vars:
    da = ds_std[v]
    print(f'  std/{v}: [{float(da.min()):.3f}, {float(da.max()):.3f}] {da.attrs.get("units","?")}')

---
## 3. Define Fixed Station Coordinates

Select N=20 fixed points from the CSV grid, biased toward the FVG coastal area (lat >= 44.5).
These coordinates are fixed for the entire pre-training and fine-tuning phases.

In [ ]:
def select_station_coords(df, n=20, coastal_lat_min=44.5, seed=42):
    coastal = df[df['lat'] >= coastal_lat_min]
    inland  = df[df['lat'] <  coastal_lat_min]
    n_coast = min(int(n * 0.75), len(coastal))
    n_inl   = n - n_coast
    sel = pd.concat([
        coastal.sample(n=n_coast, random_state=seed),
        inland.sample(n=n_inl,   random_state=seed)
    ])
    return sel[['lat', 'lon']].values   # (N, 2)

STATION_COORDS = select_station_coords(df_csv, n=N_STATION_POINTS)
print(f'Station coords shape: {STATION_COORDS.shape}')

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df_csv['lon'], df_csv['lat'], s=3, c='lightblue', alpha=0.4, label='ERA5 grid')
ax.scatter(STATION_COORDS[:, 1], STATION_COORDS[:, 0],
           s=80, c='red', zorder=5, label=f'Fixed stations (N={N_STATION_POINTS})')
ax.set(xlim=(LON_MIN, LON_MAX), ylim=(LAT_MIN, LAT_MAX),
       xlabel='Longitude', ylabel='Latitude',
       title='Fixed Station Coordinates — Pre-training')
ax.legend(); plt.tight_layout(); plt.show()

---
## 4. Normalisation

In [ ]:
class Normalizer:
    """Per-variable z-score normalisation. Save stats for inference reuse."""
    def __init__(self):
        self.stats = {}

    def fit(self, data_dict):
        for name, arr in data_dict.items():
            m, s = float(np.nanmean(arr)), float(np.nanstd(arr)) + 1e-8
            self.stats[name] = (m, s)
            print(f'  {name}: mean={m:.4f}  std={s:.4f}')

    def transform(self, name, arr):
        m, s = self.stats[name]
        return (arr - m) / s

    def inverse_transform(self, name, arr):
        m, s = self.stats[name]
        return arr * s + m


t_coord = 'valid_time' if 'valid_time' in ds_std.coords else 'time'

print('Input normaliser (ERA5 standard):')
norm_input = Normalizer()
norm_input.fit({v: ds_std[v].values.flatten() for v in VARS_INPUT if v in ds_std})

print('\nTarget normaliser (ERA5-Land):')
norm_target = Normalizer()
norm_target.fit({v: ds_land[v].values.flatten() for v in VARS_TARGET if v in ds_land})

---
## 5. PyTorch Dataset

In [ ]:
class SetConvPretrainDataset(Dataset):
    """
    Pre-training dataset for the SetConv Encoder.

    Each sample:
      sparse_input : (N_stations, 2 + C_in)  lat/lon + ERA5 standard values
      target_grid  : (C_out, H, W)            ERA5-Land grid
    """

    def __init__(self, ds_std, ds_land, station_coords,
                 vars_input, vars_target, norm_input, norm_target):

        self.station_coords = station_coords
        self.vars_input     = vars_input
        self.vars_target    = vars_target
        self.norm_input     = norm_input
        self.norm_target    = norm_target

        t_coord = 'valid_time' if 'valid_time' in ds_std.coords else 'time'
        times_std  = ds_std[t_coord].values
        times_land = ds_land[t_coord].values
        self.times = np.intersect1d(times_std, times_land)
        print(f'Shared timestamps: {len(self.times)}')

        # Pre-load all into RAM
        self.std_arrays  = {v: ds_std[v].sel({t_coord: self.times}).values
                            for v in vars_input if v in ds_std}
        self.land_arrays = {v: ds_land[v].sel({t_coord: self.times}).values
                            for v in vars_target if v in ds_land}

        self.std_lats = ds_std['latitude'].values
        self.std_lons = ds_std['longitude'].values

        # Normalise station positions to [0,1]
        self.lat_norm = (station_coords[:, 0] - LAT_MIN) / (LAT_MAX - LAT_MIN)
        self.lon_norm = (station_coords[:, 1] - LON_MIN) / (LON_MAX - LON_MIN)

    def __len__(self):
        return len(self.times)

    def _extract_at_stations(self, t_idx):
        """Nearest-neighbour extraction of ERA5 standard at station coords."""
        out = np.zeros((len(self.station_coords), len(self.vars_input)), dtype=np.float32)
        for vi, v in enumerate(self.vars_input):
            if v not in self.std_arrays:
                continue
            field = self.std_arrays[v][t_idx]    # (H, W)
            for si, (lat, lon) in enumerate(self.station_coords):
                i = int(np.argmin(np.abs(self.std_lats - lat)))
                j = int(np.argmin(np.abs(self.std_lons - lon)))
                raw = float(field[i, j])
                out[si, vi] = float(self.norm_input.transform(v, np.array([raw]))[0])
        return out

    def _build_target(self, t_idx):
        """Stack ERA5-Land channels into (C, H, W)."""
        channels = []
        for v in self.vars_target:
            if v in self.land_arrays:
                f = self.land_arrays[v][t_idx].astype(np.float32)
                channels.append(self.norm_target.transform(v, f))
        return np.stack(channels, axis=0)

    def __getitem__(self, idx):
        vals = self._extract_at_stations(idx)                        # (N, C_in)
        pos  = np.stack([self.lat_norm, self.lon_norm], axis=1)      # (N, 2)
        x    = np.concatenate([pos, vals], axis=1)                   # (N, 2+C_in)
        y    = self._build_target(idx)                               # (C_out, H, W)
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


dataset = SetConvPretrainDataset(
    ds_std=ds_std, ds_land=ds_land,
    station_coords=STATION_COORDS,
    vars_input=VARS_INPUT, vars_target=VARS_TARGET,
    norm_input=norm_input, norm_target=norm_target
)

x0, y0 = dataset[0]
print(f'Sample input  shape: {x0.shape}  (N_stations, 2+C_in)')
print(f'Sample target shape: {y0.shape} (C_out, H, W)')

---
## 6. Visualise One Sample

In [ ]:
fig, axes = plt.subplots(1, N_TARGET_VARS + 1, figsize=(5*(N_TARGET_VARS+1), 4))

# Sparse input
sc = axes[0].scatter(
    x0[:, 1].numpy() * (LON_MAX-LON_MIN) + LON_MIN,
    x0[:, 0].numpy() * (LAT_MAX-LAT_MIN) + LAT_MIN,
    c=x0[:, 2].numpy(), cmap='RdYlBu_r', s=120, zorder=5)
plt.colorbar(sc, ax=axes[0], label='msl (norm)')
axes[0].set(title='Sparse Input\n(colour=msl norm)', xlabel='Lon', ylabel='Lat')

# Target channels
for i, v in enumerate(VARS_TARGET):
    im = axes[i+1].imshow(y0[i].numpy(), origin='upper', cmap='RdYlBu_r',
                          extent=[LON_MIN,LON_MAX,LAT_MIN,LAT_MAX], aspect='auto')
    plt.colorbar(im, ax=axes[i+1], label=f'{v} (norm)')
    axes[i+1].set(title=f'Target: {v}\n(ERA5-Land)', xlabel='Lon', ylabel='Lat')

plt.suptitle('Sample 0 — Input vs Target', fontsize=12)
plt.tight_layout(); plt.show()

---
## 7. SetConv Model

```
sparse (B, N, 2+C_in)
      │
  PointEncoder (MLP per point)
      │ (B, N, hidden)
  GaussianPooling (distance-weighted aggregation)
      │ (B, hidden, H, W)
  GridDecoder (Conv2D)
      │
grid (B, C_out, H, W)
```

In [ ]:
class PointEncoder(nn.Module):
    def __init__(self, in_dim, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden)
        )
    def forward(self, x): return self.net(x)   # (B, N, hidden)


class GaussianPooling(nn.Module):
    """
    Aggregates sparse point features onto a regular grid using
    Gaussian distance-weighted averaging.
    """
    def __init__(self, H, W, sigma=0.15):
        super().__init__()
        self.H, self.W, self.sigma = H, W, sigma
        lats = torch.linspace(0, 1, H)
        lons = torch.linspace(0, 1, W)
        g_lat, g_lon = torch.meshgrid(lats, lons, indexing='ij')
        self.register_buffer('grid_pos', torch.stack([g_lat, g_lon], dim=-1))  # (H,W,2)

    def forward(self, point_pos, point_feat):
        # point_pos:  (B, N, 2)
        # point_feat: (B, N, D)
        B, N, D = point_feat.shape
        grid = self.grid_pos.view(-1, 2)                           # (H*W, 2)
        diff = point_pos.unsqueeze(2) - grid.unsqueeze(0).unsqueeze(0)  # (B,N,H*W,2)
        w = torch.exp(-(diff**2).sum(-1) / (2*self.sigma**2))     # (B,N,H*W)
        w = w / (w.sum(1, keepdim=True) + 1e-8)
        pooled = torch.bmm(point_feat.permute(0,2,1), w)          # (B,D,H*W)
        return pooled.view(B, D, self.H, self.W)


class GridDecoder(nn.Module):
    def __init__(self, hidden, out_ch, ks=5):
        super().__init__()
        p = ks//2
        self.net = nn.Sequential(
            nn.Conv2d(hidden, hidden, ks, padding=p), nn.ReLU(),
            nn.Conv2d(hidden, hidden//2, ks, padding=p), nn.ReLU(),
            nn.Conv2d(hidden//2, out_ch, 1)
        )
    def forward(self, x): return self.net(x)


class SetConvEncoder(nn.Module):
    def __init__(self, in_dim, hidden, out_ch, H, W, sigma=0.15, ks=5):
        super().__init__()
        self.pt_enc  = PointEncoder(in_dim, hidden)
        self.pool    = GaussianPooling(H, W, sigma)
        self.decoder = GridDecoder(hidden, out_ch, ks)

    def forward(self, x):
        # x: (B, N, 2+C_in)
        pos  = x[:, :, :2]              # (B, N, 2)
        feat = self.pt_enc(x)           # (B, N, hidden)
        grid = self.pool(pos, feat)     # (B, hidden, H, W)
        return self.decoder(grid)       # (B, C_out, H, W)


# Infer H, W from ERA5-Land
t_coord_land = 'valid_time' if 'valid_time' in ds_land.coords else 'time'
t0_land = ds_land[t_coord_land].values[0]
GRID_H, GRID_W = ds_land['t2m'].sel({t_coord_land: t0_land}).shape
print(f'Target grid: H={GRID_H}, W={GRID_W}')

model = SetConvEncoder(
    in_dim=2+N_INPUT_VARS, hidden=HIDDEN_DIM,
    out_ch=N_TARGET_VARS, H=GRID_H, W=GRID_W,
    sigma=SIGMA, ks=KERNEL_SIZE
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTrainable parameters: {n_params:,}')

---
## 8. Training Loop

In [ ]:
# shuffle=True: each timestep is independent at Encoder stage
loader    = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
loss_fn   = nn.MSELoss()

train_losses = []
print(f'Training for {N_EPOCHS} epochs  |  {len(loader)} batches/epoch')
print('-'*50)

for epoch in range(N_EPOCHS):
    model.train()
    ep_loss = 0.0
    for xi, yi in loader:
        xi, yi = xi.to(DEVICE), yi.to(DEVICE)
        optimizer.zero_grad()
        pred = model(xi)
        loss = loss_fn(pred, yi)
        loss.backward()
        optimizer.step()
        ep_loss += loss.item()
    avg = ep_loss / len(loader)
    train_losses.append(avg)
    scheduler.step(avg)
    if (epoch+1) % 5 == 0 or epoch == 0:
        lr = optimizer.param_groups[0]['lr']
        print(f'Epoch [{epoch+1:3d}/{N_EPOCHS}]  Loss={avg:.6f}  LR={lr:.2e}')

print('\nPre-training complete.')

---
## 9. Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, N_EPOCHS+1), train_losses, 'b-o', markersize=3, linewidth=1.5)
ax.set(xlabel='Epoch', ylabel='MSE Loss (normalised)',
       title='SetConv Pre-training — ERA5 standard → ERA5-Land')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Initial loss : {train_losses[0]:.6f}')
print(f'Final loss   : {train_losses[-1]:.6f}')
print(f'Reduction    : {(1-train_losses[-1]/train_losses[0])*100:.1f}%')

---
## 10. Qualitative Evaluation — Predicted vs Target

In [ ]:
model.eval()
with torch.no_grad():
    xi_e, yi_e = dataset[0]
    pred_e = model(xi_e.unsqueeze(0).to(DEVICE)).squeeze(0).cpu()

fig, axes = plt.subplots(N_TARGET_VARS, 3, figsize=(15, 5*N_TARGET_VARS))
if N_TARGET_VARS == 1: axes = axes[np.newaxis, :]

print('Per-variable evaluation (physical units):')
for i, v in enumerate(VARS_TARGET):
    tgt = yi_e[i].numpy()
    prd = pred_e[i].numpy()
    err = prd - tgt
    vmin, vmax = min(tgt.min(), prd.min()), max(tgt.max(), prd.max())
    emax = np.abs(err).max()

    for ax, data, title, cmap, vlo, vhi in [
        (axes[i,0], tgt, f'{v} — TARGET',     'RdYlBu_r', vmin, vmax),
        (axes[i,1], prd, f'{v} — PREDICTION', 'RdYlBu_r', vmin, vmax),
        (axes[i,2], err, f'{v} — ERROR',       'bwr',     -emax, emax),
    ]:
        im = ax.imshow(data, origin='upper', cmap=cmap, vmin=vlo, vmax=vhi,
                       extent=[LON_MIN,LON_MAX,LAT_MIN,LAT_MAX], aspect='auto')
        plt.colorbar(im, ax=ax)
        ax.set(title=title, xlabel='Lon', ylabel='Lat')

    # Physical-unit metrics
    tgt_phys = norm_target.inverse_transform(v, tgt)
    prd_phys = norm_target.inverse_transform(v, prd)
    unit = 'C' if v=='t2m' else 'mm'
    mae  = float(np.mean(np.abs(prd_phys - tgt_phys)))
    rmse = float(np.sqrt(np.mean((prd_phys - tgt_phys)**2)))
    print(f'  {v}: MAE={mae:.4f} {unit}  RMSE={rmse:.4f} {unit}')

plt.suptitle('SetConv — Qualitative Evaluation (Timestep 0)', fontsize=13)
plt.tight_layout(); plt.show()

---
## 11. Save Checkpoint

In [ ]:
ckpt = {
    'model_state_dict' : model.state_dict(),
    'model_config'     : dict(in_dim=2+N_INPUT_VARS, hidden=HIDDEN_DIM,
                              out_ch=N_TARGET_VARS, H=GRID_H, W=GRID_W,
                              sigma=SIGMA, ks=KERNEL_SIZE),
    'vars_input'        : VARS_INPUT,
    'vars_target'       : VARS_TARGET,
    'norm_input_stats'  : norm_input.stats,
    'norm_target_stats' : norm_target.stats,
    'station_coords'    : STATION_COORDS,
    'lat_range'         : (LAT_MIN, LAT_MAX),
    'lon_range'         : (LON_MIN, LON_MAX),
    'final_loss'        : train_losses[-1],
    'n_epochs'          : N_EPOCHS,
}
torch.save(ckpt, 'setconv_pretrained.pt')
print('Checkpoint saved: setconv_pretrained.pt')
print('Keys:', list(ckpt.keys()))

---
## 12. Next Steps — Fine-tuning with DPC FVG Stations

Load the checkpoint and fine-tune on real DPC station CSV data.
The input format must match exactly:

| Requirement | Value |
|---|---|
| Input shape | `(B, N_stations, 2+C_in)` |
| Lat/lon | Normalised to [0,1] with same `LAT_MIN/MAX`, `LON_MIN/MAX` |
| Variable order | `msl, u10, v10, i10fg` |
| Units | `hPa, m/s, m/s, m/s` |
| Normalisation | Use `norm_input.stats` from checkpoint |

DPC → ERA5 variable mapping (from Parameters CrossMatch):
- `B` → `msl` (hPa)
- `Vv` → wind speed (m/s) → decompose to u/v components if available
- `VvMax` → `i10fg` (m/s)

```python
# Fine-tuning skeleton (02_encoder_finetuning.ipynb)
ckpt  = torch.load('setconv_pretrained.pt')
model = SetConvEncoder(**ckpt['model_config']).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])

# Lower LR for fine-tuning
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
```